In [1]:
import pandas as pd
import numpy as np
import os
import torch



In [2]:
training_files = pd.Series(os.listdir('.'))

In [3]:
training_files = training_files[training_files.str.contains('parquet')]

In [4]:
frames = [pd.read_parquet(x) for x in training_files]

In [5]:
print([x.info() for x in frames])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63865 entries, 0 to 63864
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    63865 non-null  object
dtypes: object(1)
memory usage: 499.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1919626 entries, 0 to 1919625
Data columns (total 1 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   text    object
dtypes: object(1)
memory usage: 14.6+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27608 entries, 0 to 27607
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    27608 non-null  object
dtypes: object(1)
memory usage: 215.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240000 entries, 0 to 239999
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    240000 non-null  object
dtypes: object(1)
memory usage: 1.8+ MB
<class 'pa

In [6]:
frames[0] = pd.concat([frames[0]] * 4, ignore_index=True)
frames[0].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255460 entries, 0 to 255459
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    255460 non-null  object
dtypes: object(1)
memory usage: 1.9+ MB


In [7]:
frames[2] = pd.concat([frames[2]] * 5, ignore_index=True)

frames[2].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138040 entries, 0 to 138039
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    138040 non-null  object
dtypes: object(1)
memory usage: 1.1+ MB


In [8]:
print([x.info() for x in frames])

dataset = pd.concat(frames, ignore_index=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255460 entries, 0 to 255459
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    255460 non-null  object
dtypes: object(1)
memory usage: 1.9+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1919626 entries, 0 to 1919625
Data columns (total 1 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   text    object
dtypes: object(1)
memory usage: 14.6+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138040 entries, 0 to 138039
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    138040 non-null  object
dtypes: object(1)
memory usage: 1.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240000 entries, 0 to 239999
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    240000 non-null  object
dtypes: object(1)
memory usage: 1.8+ MB
<cla

In [9]:
# augment with more examples of patient summaries

In [10]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4981787 entries, 0 to 4981786
Data columns (total 2 columns):
 #   Column  Dtype  
---  ------  -----  
 0   text    object 
 1   label   float64
dtypes: float64(1), object(1)
memory usage: 76.0+ MB


In [11]:
shuffled_ds = dataset.sample(frac=1.0, random_state=42)

In [12]:
shuffled_ds = shuffled_ds.reset_index(drop=True)

In [13]:
shuffled_ds.to_parquet('all_training_data.parquet')

In [14]:
shuffled_ds.shape

(4981787, 2)

In [15]:
from datasets import load_dataset, Dataset
import pandas as pd

In [16]:
#shuffled_ds = pd.read_parquet('all_training_data.parquet')

In [ ]:
import pandas as pd
from datasets import Dataset

#hf_ds = Dataset.from_pandas(shuffled_ds)
hf_ds = Dataset.from_parquet('all_training_data.parquet')





from transformers import AutoTokenizer

# Load your dataset

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

# Define a tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], max_length=13000, truncation=True, padding="max_length")

# Apply tokenization
tokenized_dataset = hf_ds.map(tokenize_function, batched=True, num_proc=100)

tokenized_dataset.save_to_disk('tokenized_training_data.dataset')

In [21]:
from datasets import load_from_disk

In [22]:
tokenized_dataset = load_from_disk('tokenized_training_data.dataset/')

Loading dataset from disk:   0%|          | 0/759 [00:00<?, ?it/s]

In [29]:
len(tokenized_dataset)

4981787

In [30]:
len(tokenized_dataset['input_ids'][0])

13000

In [25]:
truncated = 0
for seq in tokenized_dataset['attention_mask'][0:10000]:
    if sum(seq) == 13000:
        truncated += 1

In [26]:
truncated

35

In [27]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

In [28]:
tokenizer.decode(tokenized_dataset['input_ids'][5050])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Nov 2025\n\nReasoning: high<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nYou are a brilliant oncologist with encyclopedic knowledge about cancer and its treatment. Your job is to evaluate whether a given clinical trial is a reasonable consideration for a patient, given a clinical trial summary and a patient summary.\n\nHere is a summary of the clinical trial:\nAge range allowed: ≥18. Sex allowed: Female. Cancer type allowed: Endometrial cancer. Histology allowed: Any. Cancer burden allowed: FIGO stage III‑IV or recurrent/metastatic disease. Prior treatment required: ≤1 line of platinum‑based chemotherapy (neo‑adjuvant, adjuvant, or concurrent) and, if progressive after platinum, a platinum‑free interval ≥12\u202fmonths. Prior treatment excluded: Immune checkpoint blocker therapy **AND** Poly (ADP‑ribose) polymerase inhibitor therapy. Biomarkers 